In [0]:
# rows = [("S1", "C1", 90.14), ("S2", None, 91.89), ("S3", "C1", 80.23)]
# columns = ["Name", "Course", "Percentage"]
# schema = StructType(
#     [
#         StructField("Name", StringType(), False), 
#         StructField("Course", StringType(), True), 
#         StructField("Percentage", DoubleType(), False)
#      ]
# )
# df = spark.createDataFrame(rows, schema=schema)

# display(df)
# df.printSchema()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# schema = StructType([
#     StructField("student_name", StringType(), False),
#     StructField("college_name", StringType(), False),
#     StructField("city", StringType(), False),
#     StructField("12th_percentage", DoubleType(), False),
#     StructField("10th_percentage", DoubleType(), False),
#     StructField("course", StringType(), True),
#     StructField("year_of_enrollment", IntegerType(), True),
#     StructField("current_semester", IntegerType(), True) 
# ])

# df = spark.read.format("csv").option("header", True).schema(schema).load("/Volumes/workspace/default/practice/student_data.csv")

# df = df.dropDuplicates(["student_name"])
# df = df.dropna()

# display(df.limit(5))
# df.count()

In [0]:
# dbutils.fs.rm("dbfs:/Volumes/workspace/default/practice/clean_data.parquet", True)
# df.write.parquet("dbfs:/Volumes/workspace/default/practice/clean_data.parquet")
# df = spark.read.parquet("/Volumes/workspace/default/practice/clean_data.parquet", header=True, inferSchema=True)
# df.count()

In [0]:

# window_spec = Window.partitionBy(F.col("college_name")).orderBy("student_name")
# df_new = df.withColumn("roll_number", F.row_number().over(window_spec))
# display(df_new.limit(5).select(["roll_number", "student_name", "college_name"]))
# df_new.write.mode("overwrite").parquet("/Volumes/workspace/default/practice/curated.parquet")


In [0]:

from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import IntegerType, StringType, DoubleType

schema = StructType([
    StructField("roll_number", IntegerType(), False),
    StructField("student_name", StringType(), False),
    StructField("college_name", StringType(), False),  
    StructField("city", StringType(), False),
    StructField("12th_percentage", DoubleType(), False),
    StructField("10th_percentage", DoubleType(), False),
    StructField("year_of_enrollment", IntegerType(), True),
    StructField("course", StringType(), True),
    StructField("current_semester", IntegerType(), True)
])

df = spark.read.parquet("/Volumes/workspace/default/practice/curated.parquet")

display(df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = df.withColumn("avg_percentage", 
                    F.round((F.col("12th_percentage") + F.col("10th_percentage")) / 2, 2)
                    )

window1 = Window.partitionBy("college_name").orderBy(F.col("avg_percentage").desc())

df = df.withColumn("college_rank", F.dense_rank().over(window1))

display(df.limit(5).filter(F.col("college_name") == "Engineering College 1").select(["roll_number","student_name", "avg_percentage", "college_rank"]))